# Setup
Notebooks call reusable functions from `src/`.


In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
from src.config import load_config, resolve_paths, set_global_seed, get_seed
config = load_config(ROOT / 'configs/project_config.yaml')
paths = resolve_paths(config)
set_global_seed(get_seed(config))
print('project root:', paths.root)


## Feature engineering

In [ ]:
from src.data_loader import load_btcirt, load_parquet, save_parquet
from src.data_validation import audit_timestamp_gaps
from src.preprocessing import preprocess
from src.feature_engineering import engineer_features, get_model_feature_columns, feature_dictionary
raw = load_btcirt(paths.raw, config)
gaps = audit_timestamp_gaps(raw.sort_values('timestamp'))
clean, meta = preprocess(raw, config, gaps)
feat = engineer_features(clean, config)
cols = get_model_feature_columns(feat)
print('n_features', len(cols))
feature_dictionary().to_csv(paths.tables/'feature_dictionary.csv', index=False)
save_parquet(feat, paths.processed)
print(feat[cols].describe().T.head())
